# 📄 Génération de corpus par domaine

**Pipeline :**
1. Charger `nodes_{domain}.csv` + `edges_{domain}.csv`
2. Former des groupes d'entités (via edges si disponibles, sinon aléatoires)
3. Générer N variantes de textes par groupe pour atteindre 2000–3000 paragraphes
4. Exporter le corpus en JSONL

**Prompt = base commun + instructions injectées selon le domaine**

## 1. Setup

In [ ]:
!pip install -q openai pandas tqdm networkx

In [ ]:
import os, json, re, time, random
from collections import defaultdict
from itertools import combinations

import pandas as pd
import networkx as nx
from openai import OpenAI
from tqdm import tqdm


# 1. Lire la clé API depuis votre fichier texte
txt_key_path = "SAMY_STUDENT_PROJECT_API_KEY.txt"  # Remplacez par le nom exact de votre fichier

try:
    with open(txt_key_path, "r", encoding="utf-8") as f:
        # .strip() est très important pour enlever les espaces ou retours à la ligne cachés
        openai_api_key = f.read().strip()
    
    # 2. Instancier le client officiel OpenAI
    client = OpenAI(api_key=openai_api_key)
    print("✅ Client OpenAI configuré avec succès depuis le fichier TXT !")
    
except FileNotFoundError:
    print(f"❌ Erreur : Le fichier '{txt_key_path}' est introuvable. Vérifiez son emplacement.")
except Exception as e:
    print(f"❌ Une erreur est survenue lors de la lecture : {e}")

MODEL = "gpt-4o-mini" 

def call_llm(messages, response_format=None, temperature=0.85, max_tokens=900):
    t0 = time.time()
    response = client.chat.completions.create(
        model       = MODEL,
        messages    = messages,
        temperature = temperature,
        max_tokens  = max_tokens,
        #service_tier = "flex",
        response_format=response_format
    )
    return {
        "text":       response.choices[0].message.content,
        "latency_ms": int((time.time() - t0) * 1000),
        "tokens":     response.usage.total_tokens,
    }

r = call_llm([{"role": "user", "content": "Reply: OK"}])
print("✅ Test LLM réussi ! Latence:", r["latency_ms"], "ms, Tokens utilisés:", r["tokens"])

## 2. Configuration des domaines

In [ ]:
# ── Paramètres globaux ─────────────────────────────────────────────────────────
TARGET_MIN = 5000   # nombre minimum de paragraphes visés
TARGET_MAX = 6000  # nombre maximum
GROUP_SIZE = 4      # entités par groupe (3–5)
MAX_RETRIES = 3
DELAY = 0.6         # secondes entre appels

# ── Instructions spécifiques par domaine ──────────────────────────────────────
# Injectées dans le prompt en plus du système commun
DOMAIN_INSTRUCTIONS = {
    "News": (
        "Write in a journalistic style, as if for a news article or press release. "
        "Use active voice, short sentences, and a neutral but engaging tone. "
        "Mention recent events, locations, or political context where relevant."
    ),
    "Job_offers": (
        "Write in a professional HR or recruitment style, as if describing a role, "
        "a company, or an industry. Mention skills, sectors, organisations, or job titles "
        "naturally. Tone should be informative and formal."
    ),
    "Movie_reviews": (
        "Write in the style of a film critic or entertainment journalist. "
        "Mention directors, actors, genres, and release context naturally. "
        "Tone can be opinionated, descriptive, and engaging."
    ),
    "Automotive_blogs": (
        "Write in the style of an automotive enthusiast blog or car review. "
        "Mention brands, models, manufacturers, countries of origin, and technical specs "
        "in a natural, passionate tone. Use industry vocabulary."
    ),
    "Amazon_product_reviews": (
        "Write in the style of a product review or consumer blog. "
        "Mention brands, companies, and products naturally, as a real user would. "
        "Tone can be informal, personal, and opinion-based."
    ),
    "Scientific_abstracts": (
        "Write in the style of a scientific abstract or academic summary. "
        "Mention researchers, institutions, fields of study, and concepts precisely. "
        "Tone should be formal, neutral, and concise."
    ),
    "Historic_blogs": (
        "Write in the style of a history blog or educational article. "
        "Mention historical figures, events, places, and periods naturally. "
        "Tone should be narrative, informative, and engaging for a general audience."
    ),
}

# Styles de variation pour générer des textes différents sur le même groupe
VARIATION_STYLES = [
    "Use a narrative storytelling angle.",
    "Focus on the relationships between the entities.",
    "Write from a historical or contextual perspective.",
    "Use an analytical, comparative tone.",
    "Write as if introducing these entities to someone unfamiliar with them.",
    "Focus on one entity in depth and mention the others in passing.",
    "Use a questioning or exploratory tone, as if investigating the topic.",
    "Write in a concise, factual summary style.",
    "Emphasize surprising or lesser-known facts about the entities.",
    "Write as if summarizing recent developments involving these entities.",
    "Take a critical or evaluative stance toward the entities.",
    "Write in first person plural ('we', 'our field', 'our industry').",
]

print(" Configuration chargée")
print(f"   Cible : {TARGET_MIN}–{TARGET_MAX} paragraphes par domaine")
print(f"   Domaines configurés : {list(DOMAIN_INSTRUCTIONS.keys())}")

## 3. Chargement des CSV et formation des groupes

In [ ]:
def load_domain_csv(domain, csv_dir="."):
    """Charge nodes_{domain}.csv et edges_{domain}.csv."""
    nodes_path = f"C:/Users/newtl/Documents/Télécom/2B/Projet Entity Linking/sous-graphes/nodes_{domain}.csv"
    edges_path = f"C:/Users/newtl/Documents/Télécom/2B/Projet Entity Linking/sous-graphes/edges_{domain}.csv"
    nodes_df = pd.read_csv(nodes_path)
    edges_df = pd.read_csv(edges_path) if os.path.exists(edges_path) else pd.DataFrame()

    # Normaliser les colonnes
    nodes = [
        {"id": row["qid"], "label": row["label"], "type": row.get("type", "entity")}
        for _, row in nodes_df.iterrows()
    ]
    edges = []
    if not edges_df.empty:
        for _, row in edges_df.iterrows():
            edges.append({
                "s": row["s"], "s_label": row["s_label"],
                "p": row["p"], "p_label": row["p_label"],
                "o": row["o"], "o_label": row["o_label"],
            })

    print(f"  {domain} → {len(nodes)} nœuds | {len(edges)} arêtes")
    return nodes, edges


def form_groups(nodes, edges, group_size=GROUP_SIZE, target_total=TARGET_MAX, seed=42):
    """
    Forme des groupes d'entités pour la génération.
    Stratégie mixte :
      1. Groupes connexes via les edges (composantes connexes ou voisinage)
      2. Groupes aléatoires pour compléter si nécessaire
    Retourne une liste de groupes, chacun étant une liste de dicts entité.
    """
    random.seed(seed)
    node_by_id = {n["id"]: n for n in nodes}
    groups = []

    # ── Groupes via edges ──────────────────────────────────────────────────────
    if edges:
        G = nx.Graph()
        G.add_nodes_from([n["id"] for n in nodes])
        for e in edges:
            if e["s"] in node_by_id and e["o"] in node_by_id:
                G.add_edge(e["s"], e["o"], label=e["p_label"])

        # Extraire des groupes à partir des composantes connexes
        for component in nx.connected_components(G):
            component = list(component)
            random.shuffle(component)
            # Découper en sous-groupes de group_size
            for i in range(0, len(component), group_size):
                chunk = component[i:i + group_size]
                if len(chunk) >= 2:  # au minimum 2 entités
                    groups.append([node_by_id[q] for q in chunk if q in node_by_id])

    # ── Groupes aléatoires pour compléter ─────────────────────────────────────
    all_ids = [n["id"] for n in nodes]
    used_ids = set(q for g in groups for e in g for q in [e["id"]])
    unused = [n for n in nodes if n["id"] not in used_ids]

    random.shuffle(unused)
    for i in range(0, len(unused), group_size):
        chunk = unused[i:i + group_size]
        if chunk:
            groups.append(chunk)

    # Si on n'a toujours pas assez de groupes pour atteindre la cible,
    # on génère des groupes aléatoires supplémentaires (avec répétition possible)
    # Le nombre de variantes par groupe sera calculé plus tard
    print(f"  → {len(groups)} groupes formés ({len([g for g in groups if len(g) >= 2])} avec 2+ entités)")
    return groups


def compute_variants_per_group(n_groups, target_min=TARGET_MIN, target_max=TARGET_MAX):
    """
    Calcule combien de variantes de texte générer par groupe
    pour atteindre la fourchette cible.
    """
    target = (target_min + target_max) // 2
    n_variants = max(1, round(target / max(n_groups, 1)))
    # Plafonner à la longueur de VARIATION_STYLES
    n_variants = min(n_variants, len(VARIATION_STYLES))
    estimated = n_groups * n_variants
    print(f"  → {n_variants} variante(s) par groupe → ~{estimated} textes estimés")
    return n_variants


print("✅ Fonctions de groupement définies")

## 4. Prompt engineering

In [ ]:
# ── Système commun ─────────────────────────────────────────────────────────────
SYSTEM_BASE = """You are an expert at generating training data for Entity Linking.

Given a list of entities (with their Wikidata IDs and types), you must:
1. Write a natural English paragraph (2–4 sentences) that mentions all or most of the entities.
2. Use VARIED and IMPRECISE mentions — like a real writer would:
   - Use only the first name OR last name instead of the full name
   - Use a pronoun after first mention (he, she, it, they)
   - Use a descriptive paraphrase (e.g. "the French director" instead of the name)
   - Abbreviate country or organization names (e.g. "US" for "United States")
   - At least HALF the entities should have imprecise/partial mentions
3. Return a JSON with the paragraph and every mention mapped to its Wikidata entity ID.

Output format (strict JSON, nothing else):
{
  "paragraph": "<the generated text>",
  "mentions": [
    {
      "mention": "<exact surface form as it appears in the paragraph>",
      "entity_id": "<Wikidata QID>",
      "entity_label": "<full canonical label>",
      "entity_type": "<type>",
      "start": <char offset start>,
      "end": <char offset end>,
      "is_partial": <true if imprecise or partial, false if exact match>
    }
  ]
}"""


def build_prompt(entities, domain, variation_style=None):
    """
    Construit le prompt utilisateur pour un groupe d'entités.
    Injecte les instructions du domaine + le style de variation.
    """
    domain_hint = DOMAIN_INSTRUCTIONS.get(domain, "")
    style_hint  = f"\nVariation style: {variation_style}" if variation_style else ""

    entity_lines = "\n".join(
        f'  - "{e["label"]}" (ID: {e["id"]}, type: {e.get("type", "entity")})'
        for e in entities
    )

    return (
        f"Domain: {domain}\n"
        f"Domain style: {domain_hint}{style_hint}\n\n"
        f"Entities to mention (vary at least half of them):\n{entity_lines}\n\n"
        f"Examples of variation: use \"Nolan\" instead of \"Christopher Nolan\", "
        f"\"the director\" instead of the name, \"US\" instead of \"United States\".\n"
        f"Return strict JSON only."
    )


def parse_output(raw_text):
    """Parse la sortie JSON du LLM avec fallback regex."""
    text = raw_text.strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*",     "", text)
    text = re.sub(r"\s*```$",     "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except Exception:
                pass
    return None


def verify_spans(paragraph, mentions):
    """Vérifie et corrige les offsets retournés par le LLM."""
    import re as _re
    verified = []
    for m in mentions:
        surface = m.get("mention", "").strip(".,!?;:'\"\'")
        if not surface:
            continue
        idx = paragraph.lower().find(surface.lower())
        if idx != -1:
            m["start"]   = idx
            m["end"]     = idx + len(surface)
            m["mention"] = paragraph[idx: idx + len(surface)]
            verified.append(m)
    return verified


print("✅ Prompts définis")

## 5. Pipeline de génération

In [ ]:
def generate_corpus_for_domain(
    domain,
    csv_dir=".",
    output_dir="output",
    target_min=TARGET_MIN,
    target_max=TARGET_MAX,
    delay=DELAY,
    max_retries=MAX_RETRIES,
    seed=42,
):
    """
    Pipeline complet pour un domaine :
      1. Charger les CSV
      2. Former les groupes
      3. Calculer le nombre de variantes
      4. Générer les textes
      5. Exporter en JSONL
    """
    os.makedirs(output_dir, exist_ok=True)
    print(f"\n{'='*60}")
    print(f"🚀 Domaine : {domain}")
    print(f"{'='*60}")

    # ── 1. Chargement ─────────────────────────────────────────────────────────
    nodes, edges = load_domain_csv(domain, csv_dir)
    if not nodes:
        print(f"  ⚠️  Aucun nœud trouvé pour {domain}, domaine ignoré.")
        return []

    # ── 2. Groupes ────────────────────────────────────────────────────────────
    groups = form_groups(nodes, edges, group_size=GROUP_SIZE, seed=seed)

    # ── 3. Nombre de variantes ────────────────────────────────────────────────
    n_variants = compute_variants_per_group(len(groups), target_min, target_max)
    styles = VARIATION_STYLES[:n_variants]

    # ── 4. Génération ─────────────────────────────────────────────────────────
    corpus, failed = [], 0
    total_calls = len(groups) * n_variants
    print(f"\n  📞 {total_calls} appels LLM prévus ({len(groups)} groupes × {n_variants} variantes)")

    pbar = tqdm(total=total_calls, desc=domain)

    for group in groups:
        for style in styles:
            prompt = build_prompt(group, domain, variation_style=style)
            success = False

            for attempt in range(max_retries):
                try:
                    result = call_llm([
                        {"role": "system", "content": SYSTEM_BASE},
                        {"role": "user",   "content": prompt},
                    ], response_format={"type": "json_object"})
                    parsed = parse_output(result["text"])

                    if not parsed or "paragraph" not in parsed:
                        raise ValueError("Invalid JSON output")

                    paragraph = parsed["paragraph"]
                    mentions  = verify_spans(paragraph, parsed.get("mentions", []))

                    if len(mentions) == 0:
                        raise ValueError("No valid mentions found")

                    corpus.append({
                        "domain":        domain,
                        "group_entities": [{"id": e["id"], "label": e["label"], "type": e.get("type", "entity")} for e in group],
                        "variation_style": style,
                        "paragraph":     paragraph,
                        "mentions":      mentions,
                        "tokens":        result["tokens"],
                        "n_partial":     sum(1 for m in mentions if m.get("is_partial")),
                        "model":         MODEL,
                    })
                    success = True
                    break

                except Exception as e:
                    wait = delay * (attempt + 1) * 2
                    tqdm.write(f"  ⚠️  Attempt {attempt+1} failed: {e} — waiting {wait:.1f}s")
                    time.sleep(wait)

            if not success:
                failed += 1

            pbar.update(1)
            time.sleep(delay)

    pbar.close()

    # ── 5. Export ─────────────────────────────────────────────────────────────
    out_path = os.path.join(output_dir, f"corpus_{domain}.jsonl")
    with open(out_path, "w", encoding="utf-8") as f:
        for record in corpus:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    n_mentions = sum(len(r["mentions"]) for r in corpus)
    n_partial  = sum(r["n_partial"]    for r in corpus)
    pct = 100 * n_partial // max(n_mentions, 1)

    print(f"\n  ✅ {len(corpus)} paragraphes générés ({failed} échecs)")
    print(f"     {n_mentions} mentions | {n_partial} partielles ({pct}%)")
    print(f"     → {out_path} ({os.path.getsize(out_path)//1024} Ko)")

    return corpus


print("✅ Pipeline défini")

In [ ]:
import os

domains = ["Historic_blogs", "Amazon_product_reviews", "Scientific_abstracts", "Movie_reviews", "News", "Job_offers", "Automotive_blogs"]
# Liste pour stocker toutes les tâches du Batch
batch_tasks = []
task_counter = 0
group_metadata = {}  # Pour stocker les métadonnées des groupes par custom_id

# Configuration de votre boucle d'origine (sur vos domaines et groupes)
# (Ici adapté selon la structure globale de votre notebook)
for domain in domains:
    nodes, edges = load_domain_csv(domain)
    groups = form_groups(nodes, edges)
    n_variants = compute_variants_per_group(len(groups), TARGET_MIN, TARGET_MAX) # Nombre de variantes par groupe, ajustable selon vos besoins
    
    for group in groups:
        # On détermine les styles à générer pour ce groupe
        selected_styles = random.sample(VARIATION_STYLES, n_variants) # Ajustez selon votre logique
        
        for style in selected_styles:
            # 1. Construire le prompt utilisateur comme vous le faisiez
            prompt_user = build_prompt(group, domain, style)
            
            # 2. Créer la structure unique pour la Batch API
            # custom_id est CRUCIAL : il doit contenir les métadonnées pour retrouver vos petits au retour !
            custom_id = f"task_{task_counter}__domain_{domain}__style_{style.replace(' ', '_')}"
            
            task = {
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": "gpt-4o-mini",
                    "messages": [
                        {"role": "system", "content": SYSTEM_BASE},
                        {"role": "user", "content": prompt_user}
                    ],
                    "temperature": 0.7,
                    "response_format": {"type": "json_object"} # On garde le format JSON strict
                }
            }
            
            # Stockage des métadonnées du groupe pour pouvoir faire le lien plus tard
            # On peut créer un dictionnaire de correspondance local (mapping)
            # ou encoder les infos essentielles dans le fichier si besoin. 
            # Le plus simple est de sauvegarder le groupe dans une liste de référence :
            group_metadata[custom_id] = group
            
            batch_tasks.append(task)
            task_counter += 1

# Écriture du fichier final au format JSONL (une ligne par dictionnaire JSON)
batch_input_path = "openai_batch_tasks.jsonl"
with open(batch_input_path, "w", encoding="utf-8") as f:
    for task in batch_tasks:
        f.write(json.dumps(task, ensure_ascii=False) + "\n")

print(f"📝 Fichier JSONL généré avec succès ! {len(batch_tasks)} requêtes prêtes à être envoyées.")

# Optionnel : Sauvegarder vos métadonnées de groupes pour l'étape de post-traitement
with open("groups_metadata.json", "w", encoding="utf-8") as f:
    json.dump(group_metadata, f, ensure_ascii=False)

In [ ]:
from openai import OpenAI

# Initialisation standard (pas besoin de gros timeout ici, l'envoi est instantané)
client = OpenAI(api_key=openai_api_key)

# 1. Téléverser le fichier local chez OpenAI
print("📤 Téléversement du fichier de requêtes...")
uploaded_file = client.files.create(
    file=open(batch_input_path, "rb"),
    purpose="batch"
)
print(f"✅ Fichier téléversé avec succès. ID : {uploaded_file.id}")

# 2. Créer le Batch Job officiel
print("🚀 Lancement du Batch Job...")
batch_job = client.batches.create(
    input_file_id=uploaded_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h" # Fenêtre d'exécution standard obligatoire
)

print(f"\n🌟 Batch lancé avec succès !")
print(f"🆔 BATCH_JOB_ID à conserver : {batch_job.id}")

In [ ]:
# Remplacez par votre ID si vous avez redémarré le notebook entre-temps
JOB_ID = batch_job.id 

status_check = client.batches.retrieve(JOB_ID)
print(f"📊 Statut : {status_check.status}")
print(f"📈 Progression : {status_check.request_counts.completed} / {status_check.request_counts.total} requêtes terminées.")

if status_check.status == "completed":
    print("🎉 Le traitement est terminé ! Vous pouvez exécuter l'étape suivante.")

In [ ]:
import json
from collections import defaultdict

# 1. Télécharger le fichier de sortie d'OpenAI
output_file_id = status_check.output_file_id
file_response = client.files.content(output_file_id)

# 2. Recharger vos métadonnées de groupes sauvegardées à l'étape 1
with open("groups_metadata.json", "r", encoding="utf-8") as f:
    group_metadata = json.load(f)

# Un dictionnaire pour regrouper les exemples par domaine : { "Movie_reviews": [...], "Amazon": [...] }
corpus_par_domaine = defaultdict(list)

print("⏳ Début du post-traitement et du tri par domaine...")

# Le fichier de retour contient une ligne JSON par requête traitée
for line in file_response.text.strip().split("\n"):
    if not line:
        continue
        
    response_data = json.loads(line)
    custom_id = response_data["custom_id"]
    
    # Extraire les métadonnées encodées dans le custom_id
    # Exemple de structure : "task_0__domain_Movie_reviews__style_Scenario"
    parts = custom_id.split("__")
    domain = parts[1].replace("domain_", "")
    style = parts[2].replace("style_", "").replace("_", " ")
    
    # Récupérer l'output textuel du LLM
    response_body = response_data["response"]["body"]
    llm_text_output = response_body["choices"][0]["message"]["content"]
    
    try:
        # Analyser le JSON renvoyé par le LLM
        parsed_llm = json.loads(llm_text_output)
        paragraph = parsed_llm.get("paragraph", "")
        raw_mentions = parsed_llm.get("mentions", [])
        
        # --- ÉTAPE DE NETTOYAGE : Votre fonction verify_spans ---
        verified_mentions = verify_spans(paragraph, raw_mentions)
        
        # Recharger dynamiquement les nœuds du domaine actuel pour générer les candidats
        nodes_list, _ = load_domain_csv(domain) 
        
        # --- ÉTAPE DES CANDIDATS : add_candidates_to_mention ---
        for mention in verified_mentions:
            true_qid = mention.get("id") or mention.get("entity_id")
            
            # Notre fonction sécurisée pour ajouter les 99 faux négatifs
            candidates = add_candidates_to_mention(true_qid, nodes_list, total_candidates=100)
            
            if candidates:
                # --- ÉTAPE DE DÉCOUPAGE (1 paragraphe = 1 mention) ---
                single_sample = {
                    "variation_style": style,
                    "paragraph": paragraph,
                    "mention": mention.get("mention"),
                    "start": mention.get("start"),
                    "end": mention.get("end"),
                    "gold_entity_id": true_qid,
                    "candidates": candidates
                }
                
                # Ajout de l'échantillon dans la liste spécifique de CE domaine
                corpus_par_domaine[domain].append(single_sample)
                
    except Exception as e:
        print(f"⚠️ Erreur lors du traitement de la tâche {custom_id} : {e}")

print("\n💾 Enregistrement des fichiers par domaine...")

# 3. Sauvegarder un fichier JSONL indépendant pour chaque domaine trouvé
for domain, items in corpus_par_domaine.items():
    # Nettoyage du nom du domaine pour le nom de fichier
    safe_domain_name = domain.replace(" ", "_").replace("/", "_")
    output_filename = f"dataset_{safe_domain_name}_flat4.jsonl"
    
    with open(output_filename, "w", encoding="utf-8") as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
            
    print(f"   ✅ {output_filename} créé avec {len(items)} exemples.")

print("\n🏁 Tout est terminé ! Les domaines ont été séparés avec succès.")

## 6. Lancement — choisir le(s) domaine(s) à traiter

In [ ]:
# ── Paramètres à ajuster ──────────────────────────────────────────────────────
CSV_DIR    = "."        # dossier contenant les nodes_*.csv et edges_*.csv
OUTPUT_DIR = "output"   # dossier de sortie

# Choisir UN domaine ou TOUS
# Option 1 — un seul domaine :
DOMAINS_TO_RUN = ["Historic_blogs"]

# Option 2 — tous les domaines :
# DOMAINS_TO_RUN = list(DOMAIN_INSTRUCTIONS.keys())

# ── Lancement ─────────────────────────────────────────────────────────────────
all_corpora = {}

for domain in DOMAINS_TO_RUN:
    corpus = generate_corpus_for_domain(
        domain,
        csv_dir    = CSV_DIR,
        output_dir = OUTPUT_DIR,
        target_min = TARGET_MIN,
        target_max = TARGET_MAX,
        delay      = DELAY,
    )
    all_corpora[domain] = corpus

print("\n" + "="*60)
print("✅ GÉNÉRATION TERMINÉE")
print("="*60)
for domain, corpus in all_corpora.items():
    print(f"  {domain:30s} → {len(corpus):5d} paragraphes")

## 7. Statistiques & aperçu

In [ ]:
import pandas as pd

for domain, corpus in all_corpora.items():
    if not corpus:
        continue

    print(f"\n{'─'*60}")
    print(f"📊 {domain}")
    print(f"{'─'*60}")

    # Stats mentions
    all_mentions = [m for r in corpus for m in r["mentions"]]
    df_m = pd.DataFrame(all_mentions)
    if not df_m.empty and "entity_type" in df_m.columns:
        print(f"Paragraphes  : {len(corpus)}")
        print(f"Mentions     : {len(df_m)}")
        print(f"Partielles   : {df_m['is_partial'].sum()} ({100*df_m['is_partial'].mean():.0f}%)")
        print()
        print("Distribution par type :")
        print(df_m["entity_type"].value_counts().to_string())

    # Aperçu d'un exemple
    sample = random.choice(corpus)
    print(f"\n📝 Exemple (style: {sample['variation_style'][:60]}...)")
    print(f"   {sample['paragraph']}")
    print(f"\n🏷  Mentions :")
    for m in sample["mentions"]:
        flag = "⚠️ PARTIAL" if m.get("is_partial") else "✅ exact"
        print(f"   [{m['start']}:{m['end']}] '{m['mention']}' → {m['entity_label']} ({m['entity_id']}) {flag}")

In [ ]:
def flatten_corpus(original_corpus):
    """
    Découpe le corpus pour obtenir un objet unique par mention.
    Si un paragraphe contient 3 mentions, il sera dupliqué 3 fois,
    chacune associée à sa mention respective et à ses 100 candidats.
    """
    flat_corpus = []
    
    for item in original_corpus:
        paragraph_text = item.get("paragraph", "")
        mentions = item.get("mentions", [])
        
        for mention in mentions:
            # On construit un nouvel objet isolé pour CETTE mention spécifique
            single_mention_data = {
                "paragraph": paragraph_text,
                # On extrait les infos de la mention unique
                "mention": mention.get("mention"),
                "start": mention.get("start"),
                "end": mention.get("end"),
                "gold_entity_id": mention.get("entity_id"), # La bonne réponse 
            }
            flat_corpus.append(single_mention_data)
            
    return flat_corpus